# PRiSM Generic Model Loading and Analysis

Loads trained models and applies the PRiSM method to generate interpretable nomograms.

## Quick Start

1. **Configure your dataset** in `.env` (same as preprocessing):
   ```bash
   PRISM_CONFIG=htx_example    # or PRISM_DATASET=my_data
   ```

2. **Set `model_prefix`** in the cell below (e.g., `mlp`, `xgb`, `logreg`)

3. **Run** to generate partial responses, LASSO selection, and nomograms

In [ ]:
%load_ext autoreload
%autoreload 2

# Set CPU fallback for MPS (Apple Silicon)
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
print("PYTORCH_ENABLE_MPS_FALLBACK =", os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK'))

import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import logging

from datetime import datetime
from prism.config import MODELS_DIR, PROCESSED_DATA_DIR
from prism.config import DATASET_PREFIX, PROJ_ROOT, CONFIG_NAME
from prism.config_loader import (  # noqa: F401
    load_config_if_exists,
    detect_target_and_id_columns,
    load_label_file,
    get_lasso_lambda_config,
    apply_lasso_lambda_selection,
    LassoConfigurationError,
    DEFAULT_TARGET_CANDIDATES,
    DEFAULT_ID_CANDIDATES,
    DEFAULT_LABEL_FILE_CANDIDATES,
)
from prism.notebook_utils import (
    validate_dataset_configured,
    load_train_test_val_data,
    load_model_checkpoint,
    load_preprocessing_metadata,
    get_analysis_params,
    load_cached_partial_responses,
    load_cached_lasso_results,
    load_cached_prn_model,
    validate_lasso_lambda_config,
    save_partial_responses_artifact,
    save_lasso_artifact,
    save_shapley_artifact,
)
from prism.logging_config import setup_logging

# =============================================================================
# DATASET CHECK - Ensure .env is configured
# =============================================================================
validate_dataset_configured(DATASET_PREFIX, CONFIG_NAME)

from prism.device_tools import get_device, get_num_cpu_workers, cleanup_gpu_memory
from prism.data_utilities import feature_summary  # noqa: F401
from prism.preprocessing import PRiSMScaler, OneHotGroupManager, collapse_onehot_features  # noqa: F401
from prism.feature_labels import FeatureLabelManager
from prism.logreg import LogisticRegression  # noqa: F401
from prism.partial_responses import partial_responses
from prism.response_plot import plot_partial_responses
from prism.nomogram_plot import nomogram, display_nomograms_side_by_side  # noqa: F401
from prism.metrics import evaluate_model_performance, compare_model_performance
from prism.maskedmlp import train_mlp, MaskedMLP
from prism.lasso import LassoRegression
from prism.wrapper import SklearnWrapper  # noqa: F401
from prism.impact import IMPACTModel, dataframe_to_tensor
from prism.shapley_plots import (
    extract_shapley_like_values,
    print_lasso_summary,
    visualize_shapley_values,
    analyze_bivariate_interactions,
)
from prism.model_persistence import save_partial_responses  # noqa: F401 (used by notebook_utils)

In [ ]:
# =============================================================================
# EXPERIMENT: Overwrite train/test targets with blackbox predictions
# =============================================================================
# When enabled, y_train and y_test are replaced with binarized blackbox outputs
# (threshold set to maintain prevalence equal to training set). Validation y is kept intact as ground truth.
# This tests how PRiSM behaves when it does not have access to the observed 
# target variable, and instead y perfectly matches the blackbox model.
OVERWRITE_Y_WITH_BLACKBOX = False

In [ ]:
# Set the model prefix to determine which trained model to load and analyze
# This determines which model files and data files will be loaded from the MODELS_DIR and PROCESSED_DATA_DIR

# model_prefix = 'impact'
# model_prefix = 'logreg'
model_prefix = 'mlp'
# model_prefix = 'xgb'
# model_prefix = 'rf'
# model_prefix = 'mlp10'

# These defaults are imported from prism.config_loader but can be overridden here.
# To customize, uncomment and modify the lines below:
# TARGET_CANDIDATES = ['my_target', 'outcome', 'label']
TARGET_CANDIDATES = list(DEFAULT_TARGET_CANDIDATES)  # Copy to allow modification

# To customize, uncomment and modify the lines below:
# ID_CANDIDATES = ['my_id', 'subject_id', 'patient_id']
ID_CANDIDATES = list(DEFAULT_ID_CANDIDATES)  # Copy to allow modification

# To customize, uncomment and modify the lines below:
# LABEL_FILE_CANDIDATES = ['my_labels.csv', 'data/labels.csv']
LABEL_FILE_CANDIDATES = list(DEFAULT_LABEL_FILE_CANDIDATES)  # Copy to allow modification

# Load configuration file if it exists
# Use CONFIG_NAME if set (from .env PRISM_CONFIG), otherwise try DATASET_PREFIX
config_to_load = CONFIG_NAME if CONFIG_NAME else DATASET_PREFIX
config = load_config_if_exists(config_to_load)

# Override manual settings with configuration file values
if config:
    print("Loaded configuration from:")
    print(f"  {PROJ_ROOT / 'example_notebooks' / 'config' / f'{config_to_load}.yaml'}")
    if 'target_candidates' in config:
        TARGET_CANDIDATES = config['target_candidates']
        print(f"  Target candidates: {TARGET_CANDIDATES}")
    if 'id_candidates' in config:
        ID_CANDIDATES = config['id_candidates']
        print(f"  ID candidates: {ID_CANDIDATES}")
    if 'label_file_candidates' in config:
        LABEL_FILE_CANDIDATES = config['label_file_candidates']
        print(f"  Label file candidates: {LABEL_FILE_CANDIDATES}")
    print("[ok] Configuration loaded and manual settings overridden")
else:
    print("No configuration file found - using manual settings")

In [ ]:
# Device: 'device' in the YAML config or the PRISM_DEVICE env var overrides
# auto-detection ('auto', 'cpu', 'cuda', 'mps'); default is the best available
device = get_device(config.get('device') if config else None)
print(f"Using device: {device}")

# Warn if CUDA was expected but unavailable (e.g., GPU assigned via CUDA_VISIBLE_DEVICES)
import os
import logging
_cuda_visible = os.environ.get('CUDA_VISIBLE_DEVICES', '')
if _cuda_visible and str(device) == 'cpu':
    _warn_msg = (
        f"WARNING: CUDA_VISIBLE_DEVICES='{_cuda_visible}' is set but CUDA is not available. "
        "This may indicate GPU driver disconnect or container GPU access loss. "
        "PRiSM analysis will run on CPU (significantly slower)."
    )
    print(_warn_msg)
    logging.getLogger(__name__).warning(_warn_msg)

# =============================================================================
# Analysis Parameters (loaded from config YAML if available, otherwise defaults)
# =============================================================================
# These settings can be overridden in the dataset config file:
#   example_notebooks/config/{DATASET_PREFIX}.yaml

# Load analysis parameters from config with sensible defaults
analysis_params = get_analysis_params(config)
random_seed = analysis_params['random_seed']
partial_response_method = analysis_params['partial_response_method']
trim_quantile = analysis_params['trim_quantile']
SAVE_NOMOGRAM_JSON = analysis_params['save_nomogram_json']
SAVE_FIGS = analysis_params['save_figs']
batch_size_scaler = analysis_params['batch_size_scaler']

# Print loaded configuration
print(f"\nAnalysis parameters:")
print(f"  random_seed: {random_seed}")
print(f"  partial_response_method: {partial_response_method}")
print(f"  trim_quantile: {trim_quantile}")
print(f"  save_nomogram_json: {SAVE_NOMOGRAM_JSON}")
print(f"  save_figs: {SAVE_FIGS}")
print(f"  batch_size_scaler: {batch_size_scaler}")

# =============================================================================
# PRN Caching Configuration
# =============================================================================
# These options allow loading cached PRN artifacts to experiment with different
# PRN lambda selection methods without re-running expensive computations.
load_cached_prn = config.get('load_cached_prn', False) if config else False
force_recalculate_prn_pr = config.get('force_recalculate_prn_partial_responses', False) if config else False
force_recalculate_prn_lasso = config.get('force_recalculate_prn_lasso', False) if config else False

if load_cached_prn:
    print(f"\nPRN caching mode enabled:")
    print(f"  load_cached_prn: {load_cached_prn}")
    print(f"  force_recalculate_prn_partial_responses: {force_recalculate_prn_pr}")
    print(f"  force_recalculate_prn_lasso: {force_recalculate_prn_lasso}")

# Set random seeds for reproducibility
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)

# Force deterministic algorithms (may reduce performance)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False
# torch.use_deterministic_algorithms(True)

In [ ]:
# Set number of workers for multithreading
max_workers = min(get_num_cpu_workers(), 32)
print(f"max_workers = {max_workers}")

In [ ]:
# Print dataset prefix information
print(f"Dataset prefix: {DATASET_PREFIX}")
print(
    f"This prefix will be used for selecting the model and data files in the pipeline from\nMODELS_DIR={MODELS_DIR}\nPROCESSED_DATA_DIR={PROCESSED_DATA_DIR}"
)

# Use model_prefix from cell 2 as the model_name
model_name = model_prefix
model_identifier = f"{DATASET_PREFIX}_{model_name}"
print(f"using model_identifier = {model_identifier}")

# Warning about model availability
print(f"\nWARNING: The model associated with prefix '{model_prefix}' must have been trained with the '{DATASET_PREFIX}' dataset.")
print(f"If the model files don't exist in {MODELS_DIR}, you need to:")
print(f"1. Run the associated preprocessing pipeline for '{DATASET_PREFIX}'")
print(f"2. Train the '{model_prefix}' model with this dataset")
print(f"3. Ensure the model files are saved before running this analysis notebook")

In [ ]:
# Set up logging (use logging.DEBUG for more detailed logs, logging.INFO for less detailed logs)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

setup_logging(
    file_log_level=logging.INFO,
    log_file=f'prism_{DATASET_PREFIX}_{model_identifier}_{timestamp}.log',
    console_output=True,
    console_log_level=logging.INFO,
)

logger = logging.getLogger(__name__)

## Load the Train/Test/Val Data

In [ ]:
# Load latest train/test/val data files
train_df, test_df, val_df, data_timestamp = load_train_test_val_data(
    model_prefix, DATASET_PREFIX, PROCESSED_DATA_DIR
)

In [ ]:
# Display basic information about the datasets
print(f"\nTrain dataset shape: {train_df.shape}")
print(f"Test dataset shape: {test_df.shape}")
print(f"Validation dataset shape: {val_df.shape}")

# Detect target and ID columns using generic detection
target_column, id_column = detect_target_and_id_columns(train_df, TARGET_CANDIDATES, ID_CANDIDATES)

if target_column:
    print(f"\nTarget column: {target_column}")
    
    # Display class distribution
    print("\nClass distribution:")
    print("Train:")
    print(train_df[target_column].value_counts(normalize=True))
    print("\nTest:")
    print(test_df[target_column].value_counts(normalize=True))
    print("\nValidation:")
    print(val_df[target_column].value_counts(normalize=True))
else:
    print("\nNo target column identified in the dataset")

if id_column:
    print(f"\nID column: {id_column}")
else:
    print("\nNo ID column identified, will create synthetic IDs if needed")

In [ ]:
# Use detected target and ID columns
# target_column and id_column are already detected above
drop_cols = [target_column, id_column] if id_column else [target_column]
drop_cols = [col for col in drop_cols if col is not None]  # Remove None values

X_train = train_df.drop(drop_cols, axis=1)
y_train = train_df[target_column]

X_test = test_df.drop(drop_cols, axis=1)
y_test = test_df[target_column]

X_val = val_df.drop(drop_cols, axis=1)
y_val = val_df[target_column]

## Load the Model and Scaler

In [ ]:
# Load model checkpoint with unified format handling
blackbox_model, scaler, hyperparameters, feature_names = load_model_checkpoint(
    model_prefix, DATASET_PREFIX, MODELS_DIR, device, X_train, random_seed
)

## Convert Data to Tensors and Scale

In [ ]:
# Transform data and convert to tensors
if model_prefix == 'impact':
    # Convert to tensors to format specifically for IMPACT model
    X_train_tensor = dataframe_to_tensor(X_train, device=device)
    X_test_tensor = dataframe_to_tensor(X_test, device=device)
    X_val_tensor = dataframe_to_tensor(X_val, device=device)
else:
    X_train_tensor = scaler.to_tensor(X_train, device=device)
    X_test_tensor = scaler.to_tensor(X_test, device=device)
    X_val_tensor = scaler.to_tensor(X_val, device=device)

# Convert target variables to tensors
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32, device=device)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32, device=device)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32, device=device)

### Feature Name Mapping

The notebook loads variable label files to map technical variable names to human-readable labels for plots and nomograms.

**Label file search order (highest to lowest priority):**
1. **Dataset-specific files**: `{dataset}_variable_labels.csv`, `{dataset}_labels.csv`, `{dataset}_label.csv`
2. **Generic fallback files**: `data_label.csv`, `generic_variable_labels.csv`, etc.

**Label file formats supported:**
- **Template format** (created by preprocessing notebook):
  - Columns: `original_name`, `processed_name`, `user_label`, `notes`
  - Maps `processed_name` → `user_label` (e.g., `DAT_AGE` → `Recipient Age (years)`)
  - Skips placeholder labels starting with `[USER:`
- **Simple format**:
  - Two columns with no header: `variable_name,Label`
  - Supports escaped newlines (`\n`) for multi-line labels

**Recommended workflow:**
1. Run preprocessing notebook to generate label template
2. Fill in the `user_label` column with descriptive names
3. Save as `{dataset}_variable_labels.csv` for dataset-specific labels
   OR keep as `generic_variable_labels.csv` for reuse across datasets with matching variable names

**Note:** If no label file is found, original variable names are used as labels.

In [ ]:
# Get feature names
if model_prefix == 'impact':
    feature_column_names = IMPACTModel.get_feature_column_names()
else:
    feature_column_names = X_train.columns.tolist()

# Load variable labels mapping using generic detection
print("Loading variable labels...")
variable_labels = load_label_file(
    LABEL_FILE_CANDIDATES, data_dir=Path('.'), dataset_prefix=DATASET_PREFIX
)
if variable_labels is None:
    # Fallback to original names
    variable_labels = {name: name for name in feature_column_names}
    print("Using original variable names as labels")

# Create FeatureLabelManager for use with plotting functions
label_manager = FeatureLabelManager(variable_labels)
print(f"Created FeatureLabelManager with {len(label_manager)} labels")

# Map feature names to descriptive labels (derived from label_manager for consistency)
feature_names = [label_manager.get_label(name) for name in feature_column_names]
# Map feature names with no line breaks (for functions that don't support multi-line labels)
feature_names_no_breaks = [label_manager.get_label(name).replace('\n', ' ') for name in feature_column_names]

print(f"Number of features: {len(feature_column_names)}")
print(f"Feature column names: {feature_column_names[:5]}... (showing first 5)")
print(f"Feature labels: {feature_names[:5]}... (showing first 5)")

# Show which labels were loaded vs using original names
loaded_labels = sum(1 for name in feature_column_names if label_manager.has_label(name))
print(f"Loaded {loaded_labels} custom labels, {len(feature_column_names) - loaded_labels} using original names")

# Show some examples of the label mapping
print("\nLabel mapping examples:")
for i, (orig, label) in enumerate(zip(feature_column_names[:3], feature_names[:3])):
    if orig != label:
        print(f"  {orig} -> {label}")
    else:
        print(f"  {orig} (no custom label)")

In [ ]:
# Load OneHotGroupManager from preprocessing metadata for collapse functionality
group_manager, preprocessing_metadata = load_preprocessing_metadata(
    DATASET_PREFIX, PROCESSED_DATA_DIR
)

# Get collapsed feature names if group manager is available
if group_manager is not None:
    X_collapsed, collapsed_feature_names = collapse_onehot_features(
        X_train.values, group_manager, feature_column_names
    )
    print(f"Collapsed feature names ({len(collapsed_feature_names)}): {collapsed_feature_names}")
else:
    collapsed_feature_names = feature_column_names

## Evaluate Black Box Performance

In [ ]:
# Calculate predicted probabilities
y_pred_train_blackbox = blackbox_model.predict(X_train_tensor)
y_pred_test_blackbox = blackbox_model.predict(X_test_tensor)

In [ ]:
if OVERWRITE_Y_WITH_BLACKBOX:
    _prevalence = float(y_train.mean())
    print(f"[EXPERIMENT] Overwriting y_train and y_test with binarized blackbox predictions")
    print(f"[EXPERIMENT] Original training prevalence: {_prevalence:.4f}")

    # Convert predictions to numpy for thresholding
    _preds_train = y_pred_train_blackbox.cpu().numpy() if torch.is_tensor(y_pred_train_blackbox) else np.asarray(y_pred_train_blackbox)
    _preds_test = y_pred_test_blackbox.cpu().numpy() if torch.is_tensor(y_pred_test_blackbox) else np.asarray(y_pred_test_blackbox)

    # Use quantile threshold to preserve original prevalence:
    # The top (prevalence * 100)% of predicted probabilities become positive
    _threshold = float(np.quantile(_preds_train.ravel(), 1.0 - _prevalence))
    print(f"[EXPERIMENT] Quantile-derived probability threshold: {_threshold:.4f}")

    _y_bin_train = (_preds_train >= _threshold).astype(int).ravel()
    _y_bin_test = (_preds_test >= _threshold).astype(int).ravel()

    print(f"[EXPERIMENT] Original  y_train prevalence: {_prevalence:.4f}")
    print(f"[EXPERIMENT] New       y_train prevalence: {_y_bin_train.mean():.4f}")
    print(f"[EXPERIMENT] Original  y_test  prevalence: {float(y_test.mean()):.4f}")
    print(f"[EXPERIMENT] New       y_test  prevalence: {_y_bin_test.mean():.4f}")

    # Overwrite pandas Series (used by LASSO, evaluation, PRN training, etc.)
    y_train = pd.Series(_y_bin_train, index=y_train.index, name=y_train.name)
    y_test = pd.Series(_y_bin_test, index=y_test.index, name=y_test.name)

    # Overwrite tensors (used by PRN training and tensor-based operations)
    y_train_tensor = torch.tensor(_y_bin_train, dtype=torch.float32, device=device)
    y_test_tensor = torch.tensor(_y_bin_test, dtype=torch.float32, device=device)

    print("[EXPERIMENT] y_train, y_test, y_train_tensor, y_test_tensor overwritten.")
    print("[EXPERIMENT] y_val and y_val_tensor are UNCHANGED (ground truth for final evaluation).")

In [ ]:
# Evaluate performance
results_blackbox_train = evaluate_model_performance(
    y_train, y_pred_train_blackbox, y_train, title=f"Blackbox ({model_prefix}) - Training Data"
)
results_blackbox_test = evaluate_model_performance(
    y_test, y_pred_test_blackbox, y_train, title=f"Blackbox ({model_prefix})- Test Data"
)

## a. Black box partial response calculation
Partial responses help us understand how individual features or pairs of features contribute to the model's predictions, making the black box model more interpretable.

In prism-only mode, cached partial responses are loaded from the source run
to avoid expensive GPU recalculation (can save hours of computation).

In [ ]:
# Set up partial response parameters
# NOTE: One-hot groups are AUTOMATICALLY collapsed when group_manager is provided
# This ensures mathematically correct reconstruction

partial_responses_params = {
    'x_train': X_train_tensor,
    'method': partial_response_method,
    'device': device,
    'batch_size': int(128 * batch_size_scaler),
    'group_manager': group_manager,
    'feature_names': feature_column_names,
    'scaler' : scaler,
}

if group_manager is not None:
    print(f"One-hot groups will be automatically collapsed: {len(collapsed_feature_names)} collapsed features")
    print(f"  Original dimensions: {len(feature_column_names)} univariate + {len(feature_column_names)*(len(feature_column_names)-1)//2} bivariate")
    print(f"  Collapsed dimensions: {len(collapsed_feature_names)} univariate + {len(collapsed_feature_names)*(len(collapsed_feature_names)-1)//2} bivariate")
else:
    print("No one-hot groups: Using original feature space")

In [ ]:
# Check for cached partial responses (prism-only mode optimization)
# Partial responses are deterministic: same model + same data = same results
# Loading cached values saves expensive GPU recalculation (can take hours)

force_recalculate = config.get('force_recalculate_partial_responses', False) if config else False
cached_pr_loaded = False
cached_pr = None  # Will hold loaded cache for val set access later
pr_dir = MODELS_DIR / 'partial_responses'

if force_recalculate:
    print("force_recalculate_partial_responses=True, will recalculate partial responses...")
elif pr_dir.exists():
    pr_pattern = f"blackbox_{model_identifier}_*_partial_responses.pt"
    pr_files = list(pr_dir.glob(pr_pattern))

    if pr_files:
        latest_pr_file = max(pr_files, key=lambda p: p.stat().st_mtime)
        print(f"Found cached partial responses: {latest_pr_file.name}")

        try:
            cached_pr = torch.load(latest_pr_file, weights_only=False)

            # Validate: model_identifier must match
            if cached_pr.get('model_identifier') != model_identifier:
                print(f"  Model mismatch: cached='{cached_pr.get('model_identifier')}', current='{model_identifier}'")
                print("  Recalculating partial responses...")
                cached_pr = None
            # Validate: sample counts must match
            elif cached_pr['train'].shape[0] != X_train_tensor.shape[0]:
                print(f"  Train count mismatch: cached={cached_pr['train'].shape[0]}, current={X_train_tensor.shape[0]}")
                print("  Recalculating partial responses...")
                cached_pr = None
            elif cached_pr['test'].shape[0] != X_test_tensor.shape[0]:
                print(f"  Test count mismatch: cached={cached_pr['test'].shape[0]}, current={X_test_tensor.shape[0]}")
                print("  Recalculating partial responses...")
                cached_pr = None
            else:
                # Load cached values - keep on CPU since partial_responses() returns CPU tensors
                # This ensures compatibility with sklearn's predict_proba calls downstream
                partial_responses_train = cached_pr['train'].cpu()
                partial_responses_test = cached_pr['test'].cpu()

                # Warn if method differs (may be intentional for experimentation)
                cached_method = cached_pr.get('partial_response_method', 'unknown')
                if cached_method != partial_response_method:
                    print(f"  NOTE: Method differs: cached='{cached_method}', config='{partial_response_method}'")
                    print("  Using cached values. Set force_recalculate_partial_responses=true to recalculate.")

                cached_pr_loaded = True
                print(f"[OK] Loaded cached blackbox partial responses")
                print(f"  Shapes: train={partial_responses_train.shape}, test={partial_responses_test.shape}")

        except Exception as e:
            print(f"  Error loading cached partial responses: {e}")
            print("  Recalculating partial responses...")
            cached_pr = None
    else:
        print(f"No cached partial responses found (pattern: {pr_pattern})")
else:
    print(f"Partial responses directory not found: {pr_dir}")

In [ ]:
# Calculate partial responses if not loaded from cache
if not cached_pr_loaded:
    print(f"Calculating partial responses using {partial_response_method} method...")
    partial_responses_train = partial_responses(X_train_tensor, blackbox_model, **partial_responses_params)

In [ ]:
# Calculate test partial responses if not loaded from cache
if not cached_pr_loaded:
    partial_responses_test = partial_responses(X_test_tensor, blackbox_model, **partial_responses_params)

In [ ]:
# [INCREMENTAL SAVE] Blackbox partial responses (train+test, val added later)
if not cached_pr_loaded:
    save_partial_responses_artifact(
        partial_responses_train, partial_responses_test,
        feature_names=feature_names_no_breaks,
        model_identifier=model_identifier,
        partial_response_method=partial_response_method,
        models_dir=MODELS_DIR,
        stage='blackbox',
        hyperparameters=hyperparameters if 'hyperparameters' in globals() else None,
    )

In [ ]:
# Free GPU memory before LASSO (minimal GPU needed for LASSO)
cleanup_gpu_memory(device)

## b. LASSO feature selection on black box partial responses

We use LASSO to select the most important partial responses. This helps identify which features or feature interactions contribute most to the model's predictions.

In [ ]:
# Determine which feature names to use for LASSO
# If one-hot groups are present (and thus automatically collapsed), use collapsed feature names
if group_manager is not None:
    lasso_feature_names = collapsed_feature_names
    # Also create labeled version for LASSO (use label_manager for consistency)
    lasso_feature_names_labeled = [label_manager.get_label(name) for name in collapsed_feature_names]
    lasso_feature_names_labeled_no_breaks = [label.replace("\n", " ") for label in lasso_feature_names_labeled]
    print(f"Using collapsed feature names for LASSO ({len(lasso_feature_names)} features)")
else:
    lasso_feature_names = feature_column_names
    lasso_feature_names_labeled = feature_names
    lasso_feature_names_labeled_no_breaks = feature_names_no_breaks
    print(f"Using original feature names for LASSO ({len(lasso_feature_names)} features)")

In [ ]:
# Check for cached LASSO results (prism-only mode optimization)
# LASSO sweep results are deterministic: same partial responses = same results
# Loading cached values allows experimenting with different lambda selection
# without re-running the expensive LASSO sweep

force_recalculate_lasso = config.get('force_recalculate_lasso', False) if config else False
cached_lasso_loaded = False
lasso_dir = MODELS_DIR / 'lasso_results'

if force_recalculate_lasso:
    print("force_recalculate_lasso=True, will recalculate LASSO...")
elif lasso_dir.exists():
    lasso_pattern = f"blackbox_{model_identifier}_*_lasso.pt"
    lasso_files = list(lasso_dir.glob(lasso_pattern))

    if lasso_files:
        latest_lasso_file = max(lasso_files, key=lambda p: p.stat().st_mtime)
        print(f"Found cached LASSO results: {latest_lasso_file.name}")

        try:
            cached_lasso = torch.load(latest_lasso_file, weights_only=False)

            # Validate: model_identifier must match
            if cached_lasso.get('model_identifier') != model_identifier:
                print(f"  Model mismatch: cached='{cached_lasso.get('model_identifier')}', current='{model_identifier}'")
                print("  Recalculating LASSO...")
            else:
                lasso_results = cached_lasso['lasso_results']
                cached_lasso_loaded = True
                print(f"[OK] Loaded cached blackbox LASSO results")
                print(f"  Lambda sweep: {len(lasso_results.lambdas)} values, {lasso_results.num_features} features")

        except Exception as e:
            print(f"  Error loading cached LASSO: {e}")
            print("  Recalculating LASSO...")
    else:
        print(f"No cached LASSO results found (pattern: {lasso_pattern})")
else:
    print(f"LASSO results directory not found: {lasso_dir}")

In [ ]:
# Set up and run LASSO regression if not loaded from cache
if not cached_lasso_loaded:
    lasso_blackbox = LassoRegression(
        nlambda=150,                # Number of lambda values to try
        min_lambda=0.001,            # Minimum regularization strength
        max_lambda=50,            # Maximum regularization strength
        batch_size=max_workers,    # Batch size for processing
        seed=random_seed,          # Random seed
        max_workers=max_workers,   # Parallel workers
        max_features=100,           # Maximum number of features to select
        base_model_name=model_prefix
    )

    print("Applying LASSO to partial responses...")
    lasso_results, fig_lasso_blackbox = lasso_blackbox.fit(
        partial_responses_train,
        partial_responses_test,
        y_train,
        y_test,
        feature_names=lasso_feature_names_labeled,
    )
else:
    # Placeholder figure for cached results
    fig_lasso_blackbox = None
    print("Skipped LASSO sweep (using cached results)")

In [ ]:
# Plot lambda path (regularization strength vs. resulting Beta coefficients)
fig_lasso_blackbox_path = lasso_results.plot_lambda_path()

In [ ]:
# Select lambda for blackbox LASSO (always run - this is configurable per run)
# Uses config from YAML if available, otherwise defaults to max_test_auc with target_ratio=0.998
# Config lookup: first tries model-specific (e.g., lasso_lambda_selection.mlp.blackbox),
#                then falls back to generic (lasso_lambda_selection.blackbox)

blackbox_lasso_config = get_lasso_lambda_config(config, 'blackbox', model=model_prefix)
apply_lasso_lambda_selection(
    lasso_results, blackbox_lasso_config,
    reference_auc=results_blackbox_test['auroc'],
)

# To manually override, comment out the lines below and call directly:
# lasso_results.select_lambda_max_test_auc(target_ratio=0.998)
# lasso_results.select_lambda_by_features(target_features=10)
# lasso_results.select_lambda_min_test_auc(min_auc=0.75)
# lasso_results.select_lambda(lambda_index=42)

In [ ]:
# [INCREMENTAL SAVE] Blackbox LASSO results
save_lasso_artifact(
    lasso_results,
    feature_names=feature_names_no_breaks,
    model_identifier=model_identifier,
    partial_response_method=partial_response_method,
    models_dir=MODELS_DIR,
    stage='blackbox',
    lambda_config=blackbox_lasso_config,
    hyperparameters=hyperparameters if 'hyperparameters' in globals() else None,
)

In [ ]:
# Plot feature importance
lasso_results.plot_beta_values()

In [ ]:
# Verify prediction breakdown for a sample
verification = lasso_results.verify_prediction(
    partial_responses_train, 
    y_train, 
    sample_idx=42,  # Sample index to analyze
    original_model_pred=y_pred_train_blackbox
)

In [ ]:
# Extract weighted Shapley-like values using LASSO selection and coefficients
shapley_results = extract_shapley_like_values(
    partial_responses_train.cpu().numpy(), 
    feature_names=feature_names_no_breaks,
    lasso_results=lasso_results,
    threshold=0.1  # LASSO selection threshold
)

# Print summary of LASSO integration
print_lasso_summary(shapley_results)

# Visualize weighted results (now shows only selected features, weighted by beta)
fig_shapley_blackbox = visualize_shapley_values(shapley_results)
plt.show()

# Analyze selected bivariate interactions
interaction_matrix = analyze_bivariate_interactions(
    partial_responses_train.cpu().numpy(),
    n_features=len(feature_names_no_breaks),
    lasso_results=lasso_results
)

In [ ]:
# [INCREMENTAL SAVE] Blackbox Shapley results
save_shapley_artifact(
    shapley_results,
    feature_names=feature_names_no_breaks,
    model_identifier=model_identifier,
    partial_response_method=partial_response_method,
    models_dir=MODELS_DIR,
    stage='blackbox',
    hyperparameters=hyperparameters if 'hyperparameters' in globals() else None,
)

## Evaluate Black Box-Nomogram Performance

In [ ]:
# Get the final Black Box-Nomogram model
blackbox_lasso = lasso_results.get_selected_model()

# Calculate predicted probabilites
y_pred_train_blackbox_lasso = blackbox_lasso.predict_proba(partial_responses_train)[:, 1]
y_pred_test_blackbox_lasso = blackbox_lasso.predict_proba(partial_responses_test)[:, 1]

In [ ]:
# Compare Black Box-Nomogram and Black Box performace
results_blackbox_vs_lasso_train = compare_model_performance(
    y_train,
    y_pred_train_blackbox,
    y_pred_train_blackbox_lasso,
    y_train=y_train,
    model_names=(f"{model_prefix}", f"{model_prefix}-Nomogram"),
    title=f"{model_prefix} vs {model_prefix}-Nomogram - Train Data",
)

results_blackbox_vs_lasso_test = compare_model_performance(
    y_test,
    y_pred_test_blackbox,
    y_pred_test_blackbox_lasso,
    y_train=y_train,
    model_names=(f"{model_prefix}", f"{model_prefix}-Nomogram"),
    title=f"{model_prefix} vs {model_prefix}-Nomogram - Test Data",
)

## c. View selected partial response plots and generate nomogram

Visualize how each selected partial response contributes to the target prediction.
Note: One-hot group collapsing is handled automatically by plot_partial_responses()
via the PlottingPipeline when group_manager is provided.

In [ ]:
# Plot partial responses (scaler and feature_names auto-handled)
# Returns: (fig_responses, fig_heatmaps) - response plots and non-mixed bivariate heatmaps
fig_responses_blackbox, fig_heatmaps_blackbox = plot_partial_responses(
    lasso_results,
    x=X_train_tensor,
    model=blackbox_model,
    scaler=scaler,  # Auto-handled internally when collapse enabled
    n_steps=50,
    method=partial_response_method,
    x_train=X_train_tensor,
    device=device,
    categorical_threshold=15,
    subtract_univariate=True,
    show_fig=True,
    return_fig=True,
    use_odds_ratio=False,
    onehot_group_manager=group_manager,
    label_manager=label_manager,  # User-friendly feature labels
    feature_names=feature_column_names,  # Pass actual column names for correct group detection
    trim_quantile=trim_quantile,
)

In [ ]:
# Generate nomogram (scaler and feature_names auto-handled)
nomogram_params = {
    'n_steps': 50,
    'method': partial_response_method,
    'x_train': X_train_tensor,
    'device': device,
    'categorical_threshold': 15,
    'subtract_univariate': True,
    'return_fig': True,
    'use_odds_ratio': False,
    'save_json': SAVE_NOMOGRAM_JSON,
    'scaler': scaler,  # Auto-handled internally when collapse enabled
    'onehot_group_manager': group_manager,
    'label_manager': label_manager,  # User-friendly feature labels
    'feature_names': feature_column_names,  # Pass actual column names for correct group detection
    'trim_quantile': trim_quantile,
    'show_conversion_line': True, 
    'legend_on_right': True,
}

result_blackbox = nomogram(
    lasso_results,
    x=X_train_tensor,
    model=blackbox_model,
    **nomogram_params
)
nomogram_main_blackbox = result_blackbox.fig_main
nomogram_non_mixed_blackbox = result_blackbox.fig_bivariate

## d. Train Partial Response Network (PRN)

Now we train a Partial Response Network based on the selected features. This network constrains the interactions to those selected using LASSO to limit spurious effects and increase stability.

In [ ]:
# Get mask from LASSO and expand if using collapsed features
mask, n_features = lasso_results.get_mask(verbose=False)

# If one-hot groups are present (automatically collapsed), expand mask from collapsed space to original one-hot space
if group_manager is not None:
    print(f"Expanding mask from collapsed ({mask.shape[0]} features) to original space...")
    mask = group_manager.expand_mask_to_original_features(mask, feature_column_names)
    print(f"Expanded mask shape: {mask.shape} (rows expanded, columns unchanged)")
    print(f"n_features remains: {n_features} (number of selected features)")

In [ ]:
scaler_prn = PRiSMScaler()

if model_prefix == 'impact':
    # Convert to tensors to format specifically for IMPACT model
    scaler_prn.fit(X_train_tensor.cpu().numpy())
    X_train_tensor_prn = scaler_prn.to_tensor(X_train_tensor.cpu().numpy())
    X_test_tensor_prn = scaler_prn.to_tensor(X_test_tensor.cpu().numpy())
    X_val_tensor_prn = scaler_prn.to_tensor(X_val_tensor.cpu().numpy())
else:
    scaler_prn.fit(X_train)
    X_train_tensor_prn = scaler_prn.to_tensor(X_train)
    X_test_tensor_prn = scaler_prn.to_tensor(X_test)
    X_val_tensor_prn = scaler_prn.to_tensor(X_val)    

### PRN Hyperparameter Tuning (Optional)

Unlike blackbox models (MLP, XGBoost, etc.), PRN hyperparameter tuning **cannot** be pre-computed
by `run_hyperparameter_tuning.py`. This is because PRN training requires the mask from LASSO
feature selection, which is only available after the blackbox model analysis.

If PRN tuning is enabled in the config YAML (`hyperparameter_tuning.prn.enabled: true`),
Optuna will run here to find optimal PRN hyperparameters. Otherwise, default hyperparameters
are used or previously saved tuned parameters are loaded.

In [ ]:
# Check if PRN tuning is enabled in config
from prism.config_loader import get_tuning_config
from prism.hyperparameter_tuning import (
    load_best_params,
    run_hyperparameter_tuning,
    save_best_params,
    save_best_model,
    load_tuned_model,
)

# Get PRN-specific tuning config
prn_tuning_config = get_tuning_config(config, 'prn')
print(f"PRN tuning enabled: {prn_tuning_config.enabled}")

tuned_prn_params = None

# Skip PRN tuning if load_cached_prn is enabled (will load cached model instead)
if load_cached_prn:
    print("Skipping PRN tuning (load_cached_prn=true, will load cached model)")
elif prn_tuning_config.enabled:
    print(f"\nRunning PRN hyperparameter tuning with {prn_tuning_config.n_trials} trials...")
    print(f"  Metric: {prn_tuning_config.metric}")
    print(f"  Direction: {prn_tuning_config.direction}")
    
    # Run Optuna tuning for PRN with the mask from LASSO selection
    tuned_prn_params, prn_study, best_prn_model = run_hyperparameter_tuning(
        model_type='prn',
        X_train=X_train_tensor_prn,
        y_train=y_train,
        X_test=X_test_tensor_prn,
        y_test=y_test,
        tuning_config=prn_tuning_config,
        random_seed=random_seed,
        device=str(device),
        mask=mask  # Pass the LASSO-selected mask
    )
    
    print(f"\nBest PRN hyperparameters (trial {prn_study.best_trial.number}):")
    for key, value in tuned_prn_params.items():
        print(f"  {key}: {value}")
    print(f"  Best {prn_tuning_config.metric}: {prn_study.best_value:.4f}")
    
    # Save tuned params for reproducibility (with model_prefix suffix to distinguish)
    prn_model_identifier = f"{DATASET_PREFIX}_{model_prefix}_prn"
    save_best_params(
        best_params=tuned_prn_params,
        model_type='prn',
        dataset_prefix=prn_model_identifier,
        output_dir=MODELS_DIR,
        study=prn_study,
        random_seed=random_seed,
        tuning_config=prn_tuning_config
    )
    print(f"Saved PRN tuned params to: {MODELS_DIR / f'{prn_model_identifier}_best_params.json'}")
    
    # Save best PRN model for reuse (skip retraining if exists)
    if best_prn_model is not None:
        save_best_model(
            model=best_prn_model,
            model_type='prn',
            dataset_prefix=prn_model_identifier,
            output_dir=MODELS_DIR,
            scaler=scaler_prn,
            hyperparameters=tuned_prn_params,
            feature_names=feature_column_names,
            mask=mask
        )
        print(f"Saved PRN tuned model to: {MODELS_DIR / prn_model_identifier / f'{prn_model_identifier}_model_tuned.pt'}")
elif getattr(prn_tuning_config, 'skip_saved_params', False):
    # skip_saved_params=True -> use defaults only
    print("skip_saved_params=True, using default PRN hyperparameters.")
else:
    # Try to load previously tuned params
    # First try model-specific PRN params (e.g., credit-g_mlp_prn)
    prn_model_identifier = f"{DATASET_PREFIX}_{model_prefix}_prn"
    tuned_prn_params = load_best_params('prn', prn_model_identifier, MODELS_DIR)
    
    # Fall back to generic PRN params if not found
    if tuned_prn_params is None:
        tuned_prn_params = load_best_params('prn', DATASET_PREFIX, MODELS_DIR)

    if tuned_prn_params:
        print("Loaded previously tuned PRN hyperparameters:")
        for key, value in tuned_prn_params.items():
            print(f"  {key}: {value}")
    else:
        print("No tuned PRN hyperparameters found. Using defaults.")

In [ ]:
# Free GPU memory before PRN training (clear accumulated blackbox computation memory)
cleanup_gpu_memory(device)

In [ ]:
# Set PRN hyperparameters - use tuned params if available, otherwise defaults
prn_hyperparameters = {
    'n_hidden': n_features,     # Number of hidden features from LASSO selection
    'mask': mask,               # Mask from LASSO selection
    'subnet_nodes': tuned_prn_params.get('subnet_nodes', 5) if tuned_prn_params else 5,
    'weight_decay': tuned_prn_params.get('weight_decay', 1e-5) if tuned_prn_params else 1e-5,
    'lr': tuned_prn_params.get('lr', 0.001) if tuned_prn_params else 0.001,
    'tolerance': 0.0001,        # Early stopping tolerance
    'patience': tuned_prn_params.get('patience', 100) if tuned_prn_params else 100,
    'max_iter': 4000,           # Maximum iterations
    'batch_size': tuned_prn_params.get('batch_size', 256) if tuned_prn_params else 256,
    'scale_lr': False,          # Don't scale LR by batch size - match tuning behavior
    'device': str(device),      # Computational device (GPU if available)
    'seed': random_seed,        # Random seed
    'plot_loss': True,          # Plot loss during training
    'plot_update_epochs': 10    # Update plot every 10 epochs
}

# Check if a tuned PRN model already exists (from hyperparameter tuning or PRN caching)
prn_model_identifier = f"{DATASET_PREFIX}_{model_prefix}_prn"
prn_model_loaded_from_cache = False

# Try loading cached PRN model if load_cached_prn is enabled
if load_cached_prn:
    cached_prn_result = load_cached_prn_model(model_identifier, model_prefix, MODELS_DIR, device)
    if cached_prn_result is not None:
        partial_response_network, prn_checkpoint = cached_prn_result
        prn_model_loaded_from_cache = True
        print(f"[OK] Loaded cached PRN model from load_cached_prn mode")
    else:
        print("Cached PRN model not found, will check for tuned model or train new one...")
        prn_checkpoint = None
else:
    prn_checkpoint = load_tuned_model('prn', prn_model_identifier, MODELS_DIR)

if not prn_model_loaded_from_cache and prn_checkpoint is not None:
    print(f"Loading tuned PRN model from: {MODELS_DIR / prn_model_identifier / f'{prn_model_identifier}_model_tuned.pt'}")
    partial_response_network = prn_checkpoint['model']
    print("Skipping PRN training - using tuned model from hyperparameter tuning.")
elif not prn_model_loaded_from_cache:
    print(f"Training PRN with {n_features} selected features...")
    partial_response_network = train_mlp(
        X_train_tensor_prn, y_train, X_test_tensor_prn, y_test, **prn_hyperparameters
    )

In [ ]:
# Plot PRN training history (from tuned checkpoint or from training)
from prism.notebook_utils import get_training_history, plot_training_history
prn_ckpt_for_history = prn_checkpoint if not prn_model_loaded_from_cache else None
prn_training_history = get_training_history(partial_response_network, prn_ckpt_for_history)
plot_training_history(prn_training_history, title_prefix='PRN')

## Evaluate PRN Performance

In [ ]:
# PRN predicted probabilities
y_pred_train_prn = partial_response_network.predict(X_train_tensor_prn)
y_pred_test_prn = partial_response_network.predict(X_test_tensor_prn)

In [ ]:
# Compare PRN to blackbox on train data
results_prn_blackbox_train = compare_model_performance(
    y_train,
    y_pred_train_blackbox,
    y_pred_train_prn,
    y_train=y_train,
    model_names=(f"{model_prefix}", "PRN"),
    title=f"{model_prefix} vs PRN - Train Data",
)

# Compare PRN to blackbox on test data
results_prn_blackbox_test = compare_model_performance(
    y_test,
    y_pred_test_blackbox,
    y_pred_test_prn,
    y_train=y_train,
    model_names=(f"{model_prefix}", "PRN"),
    title=f"{model_prefix} vs PRN - Test Data",
)

## e. PRN partial response calculation

We perform LASSO again, this time on the Partial Response Network to further isolate the most important features and interactions.

In [ ]:
# Calculate partial responses for the PRN model
partial_responses_params_prn = {
    'x_train': X_train_tensor_prn,
    'method': partial_response_method,
    'device': device,
    'batch_size': int(32 * batch_size_scaler),
    'group_manager': group_manager,
    'feature_names': feature_column_names,
    'scaler' : scaler_prn,
}

# Check for cached PRN partial responses if load_cached_prn is enabled
prn_pr_loaded_from_cache = False
if load_cached_prn and not force_recalculate_prn_pr:
    cached_prn_pr = load_cached_partial_responses(
        model_identifier, 'prn', MODELS_DIR,
        X_train_tensor_prn.shape[0], X_test_tensor_prn.shape[0]
    )
    if cached_prn_pr is not None:
        partial_responses_train_prn = cached_prn_pr['train']
        partial_responses_test_prn = cached_prn_pr['test']
        prn_pr_loaded_from_cache = True
        print(f"[OK] Loaded cached PRN partial responses")
    else:
        print("Cached PRN partial responses not found, will recalculate...")

if not prn_pr_loaded_from_cache:
    print("Calculating partial responses for PRN...")
    partial_responses_train_prn = partial_responses(
        X_train_tensor_prn, partial_response_network, **partial_responses_params_prn
    )

In [ ]:
# Calculate test PRN partial responses if not loaded from cache
if not prn_pr_loaded_from_cache:
    partial_responses_test_prn = partial_responses(
        X_test_tensor_prn, partial_response_network, **partial_responses_params_prn
    )

In [ ]:
# [INCREMENTAL SAVE] PRN partial responses (train+test, val added later)
if not prn_pr_loaded_from_cache:
    save_partial_responses_artifact(
        partial_responses_train_prn, partial_responses_test_prn,
        feature_names=feature_names_no_breaks,
        model_identifier=model_identifier,
        partial_response_method=partial_response_method,
        models_dir=MODELS_DIR,
        stage='prn',
        hyperparameters=prn_hyperparameters if 'prn_hyperparameters' in globals() else None,
    )

## f. LASSO feature selection on PRN partial responses

In [ ]:
# Check for cached PRN LASSO results if load_cached_prn is enabled
prn_lasso_loaded_from_cache = False
prn_lasso_config = get_lasso_lambda_config(config, 'prn', model=model_prefix)

if load_cached_prn and not force_recalculate_prn_lasso:
    # Note: We do NOT validate lambda config for PRN LASSO (only for blackbox)
    # because users may want to experiment with different PRN lambda selections
    cached_prn_lasso = load_cached_lasso_results(
        model_identifier, 'prn', MODELS_DIR,
        current_lambda_config=None,  # Don't validate - allow experimenting
        validate_lambda_config=False
    )
    if cached_prn_lasso is not None:
        lasso_results_prn, _ = cached_prn_lasso
        prn_lasso_loaded_from_cache = True
        print(f"[OK] Loaded cached PRN LASSO results")
    else:
        print("Cached PRN LASSO not found, will run LASSO sweep...")

if not prn_lasso_loaded_from_cache:
    # Apply LASSO to PRN partial responses
    lasso_prn = LassoRegression(
        nlambda=75,
        min_lambda=0.05,
        max_lambda=50,
        batch_size=4,
        seed=random_seed,
        max_workers=get_num_cpu_workers(),
        max_features=60,
        max_iter=2000,
        base_model_name=f'{model_prefix}_prn'
    )

    print("Applying LASSO to PRN partial responses...")
    lasso_results_prn, fig_lasso_prn = lasso_prn.fit(
        partial_responses_train_prn,
        partial_responses_test_prn,
        y_train,
        y_test,
        feature_names=lasso_feature_names_labeled,
    )
else:
    fig_lasso_prn = None
    print("Skipped LASSO sweep (using cached results)")

In [ ]:
# Plot lambda path for PRN LASSO
fig_lasso_prn_path = lasso_results_prn.plot_lambda_path()

In [ ]:
# Select lambda for PRN LASSO (always run - this allows experimenting with different selections)
# Uses config from YAML if available, otherwise defaults to max_test_auc with target_ratio=0.998
# Config lookup: first tries model-specific (e.g., lasso_lambda_selection.mlp.prn),
#                then falls back to generic (lasso_lambda_selection.prn)

apply_lasso_lambda_selection(
    lasso_results_prn, prn_lasso_config,
    reference_auc=results_blackbox_test['auroc'],
)

# To manually override, comment out the lines below and call directly:
# lasso_results_prn.select_lambda_max_test_auc(target_ratio=0.998)
# lasso_results_prn.select_lambda_by_features(target_features=10)
# lasso_results_prn.select_lambda_min_test_auc(min_auc=0.75)
# lasso_results_prn.select_lambda(lambda_index=12)

In [ ]:
# [INCREMENTAL SAVE] PRN LASSO results
save_lasso_artifact(
    lasso_results_prn,
    feature_names=feature_names_no_breaks,
    model_identifier=model_identifier,
    partial_response_method=partial_response_method,
    models_dir=MODELS_DIR,
    stage='prn',
    lambda_config=prn_lasso_config,
    hyperparameters=prn_hyperparameters if 'prn_hyperparameters' in globals() else None,
)

In [ ]:
# Plot feature importance for PRN LASSO
lasso_results_prn.plot_beta_values()

In [ ]:
# Verify prediction
verification_prn = lasso_results_prn.verify_prediction(
    partial_responses_train_prn, 
    y_train, 
    sample_idx=42,  # Sample index to analyze
    original_model_pred=y_pred_train_prn
)

In [ ]:
# Extract weighted Shapley-like values using LASSO selection and coefficients
shapley_results_prn = extract_shapley_like_values(
    partial_responses_train_prn.cpu().numpy(),
    feature_names=feature_names_no_breaks,
    lasso_results=lasso_results_prn,
    threshold=0.1,  # LASSO selection threshold
)

# Print summary of LASSO integration
print_lasso_summary(shapley_results_prn)

# Visualize weighted results (now shows only selected features, weighted by beta)
fig_shapley_prn_nomogram = visualize_shapley_values(shapley_results_prn)
plt.show()

# Analyze selected bivariate interactions
interaction_matrix = analyze_bivariate_interactions(
    partial_responses_train_prn.cpu().numpy(),
    n_features=len(feature_names_no_breaks),
    lasso_results=lasso_results_prn,
)

In [ ]:
# [INCREMENTAL SAVE] PRN Shapley results
save_shapley_artifact(
    shapley_results_prn,
    feature_names=feature_names_no_breaks,
    model_identifier=model_identifier,
    partial_response_method=partial_response_method,
    models_dir=MODELS_DIR,
    stage='prn',
    hyperparameters=prn_hyperparameters if 'prn_hyperparameters' in globals() else None,
)

## Evaluate PRN-Nomogram Performance

In [ ]:
# Get the final PRN-Nomogram model
prn_lasso = lasso_results_prn.get_selected_model()

# Calculate predicted probabilites
y_pred_train_prn_lasso = prn_lasso.predict_proba(partial_responses_train_prn)[:, 1]
y_pred_test_prn_lasso = prn_lasso.predict_proba(partial_responses_test_prn)[:, 1]

In [ ]:
# Compare PRN-Nomogram and PRN performace
results_prn_vs_lasso_train = compare_model_performance(
    y_train,
    y_pred_train_prn,
    y_pred_train_prn_lasso,
    y_train=y_train,
    model_names=(f"PRN", f"PRN-Nomogram"),
    title=f"PRN vs PRN-Nomogram - Train Data",
)

results_prn_vs_lasso_test = compare_model_performance(
    y_test,
    y_pred_test_prn,
    y_pred_test_prn_lasso,
    y_train=y_train,
    model_names=(f"PRN", f"PRN-Nomogram"),
    title=f"PRN vs PRN-Nomogram - Test Data",
)

## g. Generate final nomogram

Generate a nomogram for the PRN, based on the features selected with LASSO. This provides a visual tool to understand the model's decisions.

In [ ]:
# Plot partial responses for PRN (scaler and feature_names auto-handled)
# Returns: (fig_responses, fig_heatmaps) - response plots and non-mixed bivariate heatmaps
fig_responses_prn, fig_heatmaps_prn = plot_partial_responses(
    lasso_results_prn,
    x=X_train_tensor_prn,
    model=partial_response_network,
    scaler=scaler_prn,  # Auto-handled internally when collapse enabled
    n_steps=50,
    method=partial_response_method,
    x_train=X_train_tensor_prn,
    device=device,
    categorical_threshold=15,
    subtract_univariate=True,
    show_fig=True,
    return_fig=True,
    use_odds_ratio=False,
    onehot_group_manager=group_manager,
    label_manager=label_manager,  # User-friendly feature labels
    feature_names=feature_column_names,  # Pass actual column names for correct group detection
    trim_quantile=trim_quantile,
)

In [ ]:
# Generate nomogram for PRN (scaler and feature_names auto-handled)
nomogram_params_prn = {
    'n_steps': 50,
    'method': partial_response_method,
    'x_train': X_train_tensor_prn,
    'device': device,
    'categorical_threshold': 15,
    'subtract_univariate': True,
    'show_fig': False,
    'return_fig': True,
    'use_odds_ratio': False,
    'save_json': SAVE_NOMOGRAM_JSON,
    'comment': "PRN Nomogram",
    'scaler': scaler_prn,  # Auto-handled internally when collapse enabled
    'onehot_group_manager': group_manager,
    'label_manager': label_manager,  # User-friendly feature labels
    'feature_names': feature_column_names,  # Pass actual column names for correct group detection
    'trim_quantile': trim_quantile,
    'show_conversion_line': True,
    'legend_on_right': True,
}

result_prn = nomogram(
    lasso_results_prn,
    x=X_train_tensor_prn,
    model=partial_response_network,
    **nomogram_params_prn
)
nomogram_main_prn = result_prn.fig_main
nomogram_non_mixed_prn = result_prn.fig_bivariate

In [ ]:
# Display nomograms side by side
_ = display_nomograms_side_by_side(
    nomograms=[
        (nomogram_main_blackbox, nomogram_non_mixed_blackbox),
        (nomogram_main_prn, nomogram_non_mixed_prn),
    ],
    titles=["Blackbox Nomogram", "PRN Nomogram"],
)

## Validate and Compare Models

In [ ]:
# Free GPU memory before val partial response calculations (clear PRN computation memory)
cleanup_gpu_memory(device)

In [ ]:
# Calculate blackbox partial responses on validation dataset
# Check if cached val partial responses exist (from prism-only mode)
if cached_pr_loaded and cached_pr is not None and 'val' in cached_pr and cached_pr['val'] is not None:
    # Validate sample count matches
    if cached_pr['val'].shape[0] == X_val_tensor.shape[0]:
        # Keep on CPU for sklearn compatibility (same as partial_responses() return value)
        partial_responses_val_blackbox = cached_pr['val'].cpu()
        print(f"[OK] Loaded cached val partial responses: {partial_responses_val_blackbox.shape}")
    else:
        print(f"Val count mismatch: cached={cached_pr['val'].shape[0]}, current={X_val_tensor.shape[0]}")
        print("Calculating val partial responses...")
        partial_responses_val_blackbox = partial_responses(X_val_tensor, blackbox_model, **partial_responses_params)
else:
    partial_responses_val_blackbox = partial_responses(X_val_tensor, blackbox_model, **partial_responses_params)

In [ ]:
# [INCREMENTAL SAVE] Re-save blackbox partial responses with val
save_partial_responses_artifact(
    partial_responses_train, partial_responses_test, partial_responses_val_blackbox,
    feature_names=feature_names_no_breaks,
    model_identifier=model_identifier,
    partial_response_method=partial_response_method,
    models_dir=MODELS_DIR,
    stage='blackbox',
    hyperparameters=hyperparameters if 'hyperparameters' in globals() else None,
)

In [ ]:
# Calculate PRN partial responses on validation dataset
partial_responses_val_prn = partial_responses(
    X_val_tensor_prn, partial_response_network, **partial_responses_params_prn
)

In [ ]:
# [INCREMENTAL SAVE] Re-save PRN partial responses with val
save_partial_responses_artifact(
    partial_responses_train_prn, partial_responses_test_prn, partial_responses_val_prn,
    feature_names=feature_names_no_breaks,
    model_identifier=model_identifier,
    partial_response_method=partial_response_method,
    models_dir=MODELS_DIR,
    stage='prn',
    hyperparameters=prn_hyperparameters if 'prn_hyperparameters' in globals() else None,
)

In [ ]:
# Calculate all predicted probabilities for the four models on val dataset_filenames

# blackbox
y_pred_val_blackbox = blackbox_model.predict(X_val_tensor, device=device)

# blackbox_lasso
y_pred_val_blackbox_lasso = blackbox_lasso.predict_proba(partial_responses_val_blackbox)[:, 1]

# prn
y_pred_val_prn = partial_response_network.predict(X_val_tensor_prn, device=device)

# prn_lasso
y_pred_val_prn_lasso = prn_lasso.predict_proba(partial_responses_val_prn)[:, 1]

In [ ]:
# Compare PRN and blackbox performance
results_blackbox_vs_prn_val = compare_model_performance(
    y_val,
    y_pred_val_blackbox,
    y_pred_val_prn,
    y_train=y_train,
    model_names=(f"{model_prefix}", "PRN"),
    title=f"{model_prefix} and PRN validation comparison",
    random_seed=random_seed,
)

In [ ]:
# Compare blackbox-Nomogram and PRN-Nomogram performance
results_blackbox_vs_prn_lasso_val = compare_model_performance(
    y_val,
    y_pred_val_blackbox_lasso,
    y_pred_val_prn_lasso,
    y_train=y_train,
    model_names=(f"{model_prefix}-Nomogram", "PRN-Nomogram"),
    title=f"{model_prefix}-Nomogram and PRN-Nomogram validation comparison",
    random_seed=random_seed,
)

## Validate with example patient

In [ ]:
# Verify prediction
verification_prn = lasso_results_prn.verify_prediction(
    partial_responses_val_prn,
    y_train,
    sample_idx=42,  # Sample index to analyze
    original_model_pred=y_pred_val_prn,
)

## Table of Model Performance

In [ ]:
# Build results dict from comparison results for the performance summary.
# Term counts use the collapsed feature space (original features, not one-hot encoded)
# and count univariate + bivariate terms separately, matching the nomogram JSON output.
from rich.console import Console
from rich.table import Table
import pandas as pd
from datetime import datetime
from prism.notebook_utils import extract_performance_data

# Collect AUROC results from all stages/splits into a single dict
_results_dict = {}

if 'results_blackbox_train' in globals():
    _results_dict[('Blackbox', 'Train')] = results_blackbox_train
if 'results_blackbox_test' in globals():
    _results_dict[('Blackbox', 'Test')] = results_blackbox_test

if 'results_blackbox_vs_lasso_train' in globals():
    _lasso_key = list(results_blackbox_vs_lasso_train.keys())[1]
    _results_dict[('Blackbox-Nomogram', 'Train')] = results_blackbox_vs_lasso_train[_lasso_key]
if 'results_blackbox_vs_lasso_test' in globals():
    _lasso_key = list(results_blackbox_vs_lasso_test.keys())[1]
    _results_dict[('Blackbox-Nomogram', 'Test')] = results_blackbox_vs_lasso_test[_lasso_key]

if 'results_prn_blackbox_train' in globals():
    _prn_key = list(results_prn_blackbox_train.keys())[1]
    _results_dict[('PRN', 'Train')] = results_prn_blackbox_train[_prn_key]
if 'results_prn_blackbox_test' in globals():
    _prn_key = list(results_prn_blackbox_test.keys())[1]
    _results_dict[('PRN', 'Test')] = results_prn_blackbox_test[_prn_key]

if 'results_prn_vs_lasso_train' in globals():
    _prn_lasso_key = list(results_prn_vs_lasso_train.keys())[1]
    _results_dict[('PRN-Nomogram', 'Train')] = results_prn_vs_lasso_train[_prn_lasso_key]
if 'results_prn_vs_lasso_test' in globals():
    _prn_lasso_key = list(results_prn_vs_lasso_test.keys())[1]
    _results_dict[('PRN-Nomogram', 'Test')] = results_prn_vs_lasso_test[_prn_lasso_key]

if 'results_blackbox_vs_prn_val' in globals():
    _bb_key = list(results_blackbox_vs_prn_val.keys())[0]
    _prn_key = list(results_blackbox_vs_prn_val.keys())[1]
    _results_dict[('Blackbox', 'Val')] = results_blackbox_vs_prn_val[_bb_key]
    _results_dict[('PRN', 'Val')] = results_blackbox_vs_prn_val[_prn_key]

if 'results_blackbox_vs_prn_lasso_val' in globals():
    _bb_lasso_key = list(results_blackbox_vs_prn_lasso_val.keys())[0]
    _prn_lasso_key = list(results_blackbox_vs_prn_lasso_val.keys())[1]
    _results_dict[('Blackbox-Nomogram', 'Val')] = results_blackbox_vs_prn_lasso_val[_bb_lasso_key]
    _results_dict[('PRN-Nomogram', 'Val')] = results_blackbox_vs_prn_lasso_val[_prn_lasso_key]

# Blackbox input features: use collapsed (original) feature space, not one-hot encoded
# This matches the nomogram perspective where each original variable is one term
n_blackbox_features = len(collapsed_feature_names) if group_manager is not None else len(feature_column_names)

performance_data, csv_data = extract_performance_data(
    _results_dict,
    lasso_results_blackbox=lasso_results,
    lasso_results_prn=lasso_results_prn,
    n_blackbox_features=n_blackbox_features,
)

# Create Rich table
console = Console()
table = Table(title="Model Performance Summary", show_header=True, header_style="bold magenta")

# Add columns
table.add_column("Dataset", style="green")
table.add_column("Model", style="cyan", no_wrap=True)
table.add_column("n_terms", justify="right", style="yellow")
table.add_column("(univ+biv)", justify="right", style="yellow")
table.add_column("AUROC", justify="right", style="blue")
table.add_column("95% CI", justify="center", style="magenta")

# Define model order and dataset order
model_order = ['Blackbox', 'Blackbox-Nomogram', 'PRN', 'PRN-Nomogram']
dataset_order = ['Train', 'Test', 'Val']

# Add rows to table
for dataset in dataset_order:
    for model in model_order:
        key = (model, dataset)
        if key in performance_data:
            data = performance_data[key]
            auroc = f"{data['auroc']:.3f}"
            ci_lower, ci_upper = data['auroc_ci']
            ci_str = f"({ci_lower:.3f}-{ci_upper:.3f})"
            terms = data['terms']
            terms_str = str(terms['n_total'])
            breakdown = f"({terms['n_univ']}+{terms['n_biv']})"
            table.add_row(dataset, model, terms_str, breakdown, auroc, ci_str)
        else:
            table.add_row(dataset, model, "N/A", "", "N/A", "N/A")

# Display the table
console.print(table)

# Create and save CSV file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = f"model_performance_summary_{model_identifier}_{timestamp}.csv"

# Create DataFrame and save to CSV
df_performance = pd.DataFrame(csv_data)
csv_path = MODELS_DIR / 'performance_summaries'
csv_path.mkdir(exist_ok=True)
full_csv_path = csv_path / csv_filename

# Add metadata as comments in CSV
with open(full_csv_path, 'w') as f:
    f.write(f"# Model Performance Summary Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"# Model Identifier: {model_identifier}\n")
    f.write(f"# Random Seed: {random_seed}\n")
    f.write(f"# Blackbox Input Features (collapsed): {n_blackbox_features}\n")

    # Add model hyperparameters if available
    if 'hyperparameters' in globals():
        f.write("# Model Hyperparameters:\n")
        for param, value in hyperparameters.items():
            f.write(f"#   {param}: {value}\n")

    # Add model class information if available
    if 'model_class' in globals():
        f.write(f"# Model Class: {model_class.__name__}\n")

    # Add device information
    f.write(f"# Device: {device}\n")

    # Add partial response method
    if 'partial_response_method' in globals():
        f.write(f"# Partial Response Method: {partial_response_method}\n")

    # Add training parameters if available
    if 'prn_hyperparameters' in globals():
        f.write("# PRN Hyperparameters:\n")
        for param, value in prn_hyperparameters.items():
            if param != 'mask':  # Skip mask as it's too large
                f.write(f"#   {param}: {value}\n")

    # Add Nomogram parameters if available
    if 'lasso_blackbox' in globals():
        f.write("# Blackbox Nomogram Parameters:\n")
        f.write(f"#   nlambda: {getattr(lasso_blackbox, 'nlambda', 'N/A')}\n")
        f.write(f"#   min_lambda: {getattr(lasso_blackbox, 'min_lambda', 'N/A')}\n")
        f.write(f"#   max_lambda: {getattr(lasso_blackbox, 'max_lambda', 'N/A')}\n")
        f.write(f"#   max_features: {getattr(lasso_blackbox, 'max_features', 'N/A')}\n")

    if 'lasso_prn' in globals():
        f.write("# PRN Nomogram Parameters:\n")
        f.write(f"#   nlambda: {getattr(lasso_prn, 'nlambda', 'N/A')}\n")
        f.write(f"#   min_lambda: {getattr(lasso_prn, 'min_lambda', 'N/A')}\n")
        f.write(f"#   max_lambda: {getattr(lasso_prn, 'max_lambda', 'N/A')}\n")
        f.write(f"#   max_features: {getattr(lasso_prn, 'max_features', 'N/A')}\n")

    f.write("#\n")

# Append the actual data
df_performance.to_csv(full_csv_path, mode='a', index=False)

print(f"\nPerformance summary saved to: {full_csv_path}")

## Save All Model Predictions

In [ ]:
# Generate timestamp for the filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create dictionary to hold all predictions
predictions = {
    'train': {
        'df': train_df.copy(),  # Use copy to avoid modifying original
        'blackbox': y_pred_train_blackbox.cpu().numpy(),
        'partial_responses': partial_responses_train,
    },
    'test': {
        'df': test_df.copy(),  # Use copy to avoid modifying original
        'blackbox': y_pred_test_blackbox.cpu().numpy(),
        'partial_responses': partial_responses_test,
    },
    'val': {
        'df': val_df.copy(),  # Use copy to avoid modifying original
        'blackbox': blackbox_model.predict(X_val_tensor).cpu().numpy(),
        'partial_responses': partial_responses_val_blackbox,
    },
}

# Get PRN predictions for all sets if PRN is available
try:
    predictions['train']['prn'] = y_pred_train_prn.cpu().numpy()
    predictions['test']['prn'] = y_pred_test_prn.cpu().numpy()
    predictions['val']['prn'] = y_pred_val_prn.cpu().numpy()
    has_prn = True
    print("PRN model found, adding PRN predictions")
except NameError:
    print("PRN model not found, skipping PRN predictions")
    has_prn = False

# Get LASSO predictions
blackbox_lasso = lasso_results.get_selected_model()
prn_lasso = None
if has_prn:
    try:
        prn_lasso = lasso_results_prn.get_selected_model()
        print("PRN LASSO model found, adding PRN LASSO predictions")
    except NameError:
        print("PRN LASSO model not found, skipping PRN LASSO predictions")

# Store names of existing PR variables
pr_var_mapping = {
    'train': {'prn': 'partial_responses_train_prn'},
    'test': {'prn': 'partial_responses_test_prn'},
    'val': {'prn': 'partial_responses_val_prn'},
}

# Add LASSO predictions
for dataset in ['train', 'test', 'val']:
    # Add blackbox LASSO predictions
    predictions[dataset]['blackbox_nomogram'] = blackbox_lasso.predict_proba(
        predictions[dataset]['partial_responses']
    )[:, 1]

    # Add PRN LASSO predictions if available
    if prn_lasso is not None and has_prn:
        pr_var_name = pr_var_mapping[dataset]['prn']
        try:
            if pr_var_name in locals():
                pr_prn = locals()[pr_var_name]
                predictions[dataset]['prn_nomogram'] = prn_lasso.predict_proba(pr_prn)[:, 1]
            else:
                # Calculate if not found
                print(f"Calculating PRN partial responses for {dataset} dataset...")
                x_tensor = locals()[f"X_{dataset}_tensor"]
                pr_prn = partial_responses(
                    x_tensor, partial_response_network, **partial_responses_params_prn
                )
                predictions[dataset]['prn_nomogram'] = prn_lasso.predict_proba(pr_prn)[:, 1]
        except Exception as e:
            print(f"Error calculating PRN LASSO predictions for {dataset}: {e}")

# Add predictions and datasplit column to dataframes
combined_dfs = []
for dataset, data in predictions.items():
    # Add all predictions to dataframe at once
    data['df']['pred_blackbox'] = data['blackbox']
    data['df']['pred_blackbox_nomogram'] = data['blackbox_nomogram']

    if has_prn and 'prn' in data:
        data['df']['pred_prn'] = data['prn']

    if has_prn and prn_lasso is not None and 'prn_nomogram' in data:
        data['df']['pred_prn_nomogram'] = data['prn_nomogram']

    # Add datasplit column
    data['df']['datasplit'] = dataset

    # Add to list for combining
    combined_dfs.append(data['df'])

# Combine all dataframes
combined_df = pd.concat(combined_dfs, ignore_index=True)

# Create a subdirectory in MODELS_DIR for our predictions
pred_dir = MODELS_DIR.joinpath('predictions')
pred_dir.mkdir(exist_ok=True)

# Save the combined dataframe with predictions
output_path = pred_dir.joinpath(f"{model_identifier}_preds_{timestamp}.csv")
combined_df.to_csv(output_path, index=False)

print(f"\nSaved combined prediction file:")
print(f"  - {output_path}")

# Print summary statistics
print(f"\nDataset summary:")
print(combined_df['datasplit'].value_counts().sort_index())

print("\nAll prediction data saved successfully in a single file.")

## Save All Figures

In [ ]:
if SAVE_FIGS:
    from prism.save_all_figures import save_all_figures

    saved_figure_paths = save_all_figures()
    print(f"Figures saved to {saved_figure_paths}")